In [ ]:
# Your API keys / URLs live in a hidden ".env" file (never shared, never committed).
# load_dotenv() reads that file so os.getenv(...) can find those values later.
from dotenv import load_dotenv
import os

load_dotenv()  # returns True if a .env file was found and loaded

# 📓 1.1 — Foundational Models

This notebook covers four things, each building on the last:

1. **Model** — the AI you talk to (here it's your `qwen3-30b`).
2. **Invoke** — send it a message, get an answer back.
3. **Agent** — a model that can work in a loop and use tools, not just reply once.
4. **Streaming** — printing the answer as it's written, word by word.

Run each cell with **Shift + Enter**. Cells share state, so a variable made up top is still around further down.

## Initialising and invoking a model

Two steps here:

- **Initialise** = set up a connection to the model once, and keep it in a variable.
- **Invoke** = actually send it a message and wait for the reply.

`init_chat_model` is LangChain's universal "give me a model" helper — the same function works for OpenAI, Claude, Gemini, or (like here) your own server. The non-obvious part is the four arguments below, so they're explained in the code.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen3-30b",             # which model to run on the server
    model_provider="openai",       # NOT the vendor — it's the *API format*. Your server speaks
                                   # the OpenAI format, so we say "openai" even though it's Qwen.
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),   # where your server lives (the /v1 URL)
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),     # your key for it, pulled from .env
)

# `model` is now a reusable connection. We haven't sent anything yet — that's the next cell.

In [ ]:
from pprint import pprint  # "pretty print" — lays big objects out over multiple lines so they're readable

# .invoke(...) sends your text and waits for the full reply.
response = model.invoke("What's the capital of the Moon?")

# `response` is NOT just a string — it's an AIMessage object. Printing it shows everything
# the model returned: the text in `.content`, plus extras like token counts and the model name.
pprint(response)

In [10]:
print(response.content)

The Moon does not have a capital, because it is not a country or a sovereign nation. It is a natural satellite of Earth and does not have a government, population, or political structure. While there are scientific research stations and rovers operated by space agencies like NASA, ESA, and others, these are not cities or capitals in the traditional sense. So, there is no capital of the Moon. 🌕🚀


In [11]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-af1377f58c770a44',
 'logprobs': None,
 'model_name': 'qwen3-30b',
 'model_provider': 'openai',
 'system_fingerprint': 'vllm-0.22.1-a8343ffa',
 'token_usage': {'completion_tokens': 86,
                 'completion_tokens_details': None,
                 'prompt_tokens': 16,
                 'prompt_tokens_details': None,
                 'total_tokens': 102}}


## Customising your Model

In [19]:
model = init_chat_model(
    model="qwen3-30b",
    model_provider= "openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

The Moon doesn't have a capital city, because it is not a country or a sovereign nation. It is a natural satellite of Earth and currently has no permanent human settlements. While there are scientific missions and plans for future lunar bases, there is no official capital. 🌕🚀


## Model Providers

https://docs.langchain.com/oss/python/integrations/chat

In [23]:
model = init_chat_model(
        model="qwen3-30b",
    model_provider= "openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),

)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

The Moon doesn't have a capital city, because it is not a country or a sovereign nation. It is a natural satellite of Earth and does not have a government, population, or cities. While there are plans by various space agencies (like NASA, ESA, and others) to establish lunar bases in the future, no official capital has been established—or even designated—on the Moon.


In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI

# model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
#
# response = model.invoke("What's the capital of the Moon?")
# print(response.content)

## Initialising and invoking an agent

In [32]:
from langchain.agents import create_agent

agent = create_agent(model=model)

In [ ]:
# agent = create_agent(model="claude-sonnet-4-5")

In [ ]:
# agent = create_agent("gpt-5-nano")

In [34]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [35]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='cf025894-f6b6-43f8-9d1e-9035bb26bd8b'),
              AIMessage(content="The Moon doesn't have a capital city, because it is not a country or a sovereign nation. It is a natural satellite of Earth and does not have a government, population, or cities. While there are plans by various space agencies (like NASA, ESA, and others) to establish lunar bases in the future, no official capital has been established—or even designated—on the Moon.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 16, 'total_tokens': 95, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-30b', 'system_fingerprint': 'vllm-0.22.1-a8343ffa', 'id': 'chatcmpl-b748f72a2a7779cf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4cf7-0630-7950-b4b6-9f4a7cf3c915-0', 

In [36]:
print(response['messages'][-1].content)

The Moon doesn't have a capital city, because it is not a country or a sovereign nation. It is a natural satellite of Earth and does not have a government, population, or cities. While there are plans by various space agencies (like NASA, ESA, and others) to establish lunar bases in the future, no official capital has been established—or even designated—on the Moon.


In [37]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='34aed696-4e30-4246-8da2-1e1c73c44929'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='847425e2-1e6a-4bc8-a766-cf0f8e296334', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='4c9cdbb9-f124-4d87-99fc-90ec759690fe'),
              AIMessage(content="Ah, Luna City—our gleaming metropolis on the Moon! 🌕🏙️  \n\nWhile the Moon currently has no official capital (as it's not a sovereign nation and has no permanent residents), *Luna City* is a popular concept in science fiction, space exploration planning, and futuristic dreams. Let’s imagine it as the heart of humanity’s lunar presence:\n\n### 🌕 **Luna City: The Future Lunar Capital (Fictional Overview)**  \n- **Location**: Mare Tranquilli

## Streaming Output

In [40]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

As of now, **Luna City does not exist**—it is not the capital of the Moon, nor is there a permanent human settlement on the Moon. However, your question touches on a fascinating topic in science fiction and future space exploration visions. Let’s explore the idea of "Luna City" in both speculative and real-world contexts.

---

### 🌕 **The Concept of Luna City (Fictional / Speculative)**
In science fiction and futuristic planning, **Luna City** often refers to a hypothetical permanent human settlement on the Moon. It's imagined as:

- **The capital of lunar civilization**, serving as the political, scientific, and cultural hub of the Moon.
- Located in a **lunar polar region** (like near the South Pole), where water ice has been detected—critical for life support and fuel production.
- Built using **in-situ resource utilization (ISRU)**: mining lunar regolith (soil) for construction materials and extracting water ice for oxygen and hydrogen.
- Protected by **underground habitats or pre